# Create Kavli Prize Awards (PRIZE PATTERN)

Kavli Prize laureates extracted from `__NEXT_DATA__` JSON embedded in
kavliprize.org/laureates. The Kavli Prize is biennial since 2008
across three fields: astrophysics, nanoscience, neuroscience.

**Awarding body:** Kavli Foundation — F4320306399

**Prerequisites:**
- Run `scripts/local/kavli_to_s3.py` first.

**S3:** `s3a://openalex-ingest/awards/kavli/kavli_laureates.parquet`

Schema notes:
- One row per (year × field × laureate). 73 rows total as of 2024.
- Each Kavli Prize is $1M USD per category, shared among the
  laureates of that category. Per-laureate amount is published as
  `prize/laureateCount` historically. We capture amount = NULL by
  default but the notebook can be extended to compute the share.
  Step 6.7 amount check is waived.
- currency = "USD".
- field is one of {astrophysics, nanoscience, neuroscience}.
- Bio (where present) goes into description.


## Step 1: Create staging table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.kavli_raw
USING delta
AS
SELECT *, current_timestamp() AS databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/kavli/kavli_laureates.parquet`;

In [ ]:
%sql
SELECT COUNT(*) FROM openalex.awards.kavli_raw;

In [ ]:
%sql
DESCRIBE openalex.awards.kavli_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.kavli_raw LIMIT 5;

## Step 2: Transform

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.kavli_awards
USING delta
AS
WITH kavli_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320306399
)
SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':kavli:', LOWER(s.kavli_laureate_id)))) % 9000000000 AS id,
    CONCAT('Kavli Prize in ', initcap(s.field), ' ', CAST(s.year AS STRING),
           ' — ', s.name) AS display_name,
    -- Description = the per-prize citation (concise, e.g. "for their ground-breaking work...").
    -- The bio paragraph is much longer biographical prose — useful but not the right description.
    NULLIF(s.citation, '') AS description,
    f.funder_id,
    s.kavli_laureate_id AS funder_award_id,
    CAST(NULL AS DOUBLE) AS amount,
    'USD' AS currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) AS id,
        f.display_name, f.ror_id, f.doi
    ) AS funder,
    'prize' AS funding_type,
    initcap(s.field) AS funder_scheme,
    'kavli_nextdata' AS provenance,
    TRY_TO_DATE(CONCAT(CAST(s.year AS STRING), '-01-01'), 'yyyy-MM-dd') AS start_date,
    TRY_TO_DATE(CONCAT(CAST(s.year AS STRING), '-12-31'), 'yyyy-MM-dd') AS end_date,
    s.year AS start_year,
    s.year AS end_year,
    struct(
        s.given_name AS given_name,
        s.family_name AS family_name,
        CAST(NULL AS STRING) AS orcid,
        CAST(NULL AS DATE) AS role_start,
        struct(
            NULLIF(s.institution, '') AS name,
            element_at(s.countries, 1) AS country,
            CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) AS ids
        ) AS affiliation
    ) AS lead_investigator,
    CAST(NULL AS STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >) AS co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >>) AS investigators,
    CONCAT('https://kavliprize.org/laureates/', COALESCE(s.slug, s.kavli_laureate_id)) AS landing_page_url,
    CAST(NULL AS STRING) AS doi,
    concat('https://api.openalex.org/works?filter=awards.id:G',
           abs(xxhash64(CONCAT(f.funder_id, ':kavli:', LOWER(s.kavli_laureate_id)))) % 9000000000) AS works_api_url,
    current_timestamp() AS created_date,
    current_timestamp() AS updated_date
FROM openalex.awards.kavli_raw s
CROSS JOIN kavli_funder f
WHERE s.kavli_laureate_id IS NOT NULL AND s.year IS NOT NULL;

## Step 3: Insert into openalex_awards_raw at priority 49

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'kavli_nextdata' AND priority = 49;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT id, display_name, description, funder_id, funder_award_id,
       amount, currency, funder, funding_type, funder_scheme, provenance,
       start_date, end_date, start_year, end_year,
       lead_investigator, co_lead_investigator, investigators,
       landing_page_url, doi, works_api_url,
       created_date, updated_date,
       49 AS priority
FROM openalex.awards.kavli_awards;

## Verification

In [ ]:
%sql
SELECT COUNT(*) total, COUNT(amount) has_amount,
       ROUND(COUNT(amount)*100.0/COUNT(*), 1) pct_amount,
       collect_set(currency) currencies,
       collect_set(funder_scheme) categories
FROM openalex.awards.kavli_awards;

In [ ]:
%sql
SELECT id, SUBSTRING(display_name, 1, 80) AS title, funder_scheme AS field,
       start_year, lead_investigator.given_name, lead_investigator.family_name,
       lead_investigator.affiliation.name AS institution
FROM openalex.awards.kavli_awards ORDER BY start_year DESC, funder_scheme LIMIT 15;

In [ ]:
%sql
SELECT funder_scheme, start_year, COUNT(*) cnt
FROM openalex.awards.kavli_awards GROUP BY funder_scheme, start_year
ORDER BY start_year DESC, funder_scheme;